**Author:** Brendan OConnell 

**Date:** May 2026  

**Purpose:** Alternative Pre-Processing pipeline (minimal steps), used for XGBoost

**Key decisions:**  

* During XGBoost baseline model eval, the removal of the scarce elements was reconsidered. By nature, tree models are designed to handle scarcity, and there may be real signal in those scarce elements. It seems there may be more risk than reward for dropping them, at least for the XGboost dataset.
* Drop ambiguous particle classes from main dataset (cannot confidently label as GSR or Non-GSR)
* Apply binary GSR/Non-GSR target label to the remaining particles

In [1]:
import pandas as pd

# Load the tidy NFI dataset

In [2]:
particle_df = pd.read_parquet("../../../data/processed/particle_labeled.parquet")
print(f"Shape: {particle_df.shape}")
print(f"Columns: {particle_df.columns}")

Shape: (2801667, 95)
Columns: Index(['stub_id', 'particle_id', 'relevance_class', 'ac', 'ag', 'al', 'ar',
       'as', 'at', 'au', 'b', 'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co',
       'cr', 'cs', 'cu', 'dy', 'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge',
       'hf', 'hg', 'ho', 'i', 'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn',
       'mo', 'n', 'na', 'nb', 'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa',
       'pb', 'pd', 'pm', 'po', 'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn',
       'ru', 's', 'sb', 'sc', 'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc',
       'te', 'th', 'ti', 'tl', 'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn',
       'zr', 'merged_relevance_class', 'final_class', 'label'],
      dtype='str')


**EDA's ambiguous particle overlay of the UMAP illustrated the nature of the ambiguous particles.**

Cannot confidently label as GSR or Non-GSR.

Drop ambiguous particles from the processed dataset.

In [3]:
binary_informative_df = particle_df[particle_df["label"] != "Ambiguous"]

**Encode the label as a binary target column: `GSR=1` / `Non-GSR=0`**

Also, drop "relevant_class" and "merged_relevant_class" meta columns. Only the "final_class" is needed for downstream processing.

In [4]:
binary_informative_df["target"] = (binary_informative_df["label"] == "GSR").astype(int)
binary_informative_df = binary_informative_df.drop(
    columns=["relevance_class", "merged_relevance_class"]
)
binary_informative_df.rename(columns={"final_class": "class"}, inplace=True)
binary_informative_df.shape

(2294985, 94)

Write the binary-labeled preprocessed data to parquet for use with Feature Engineering

In [5]:
# binary_informative_df.to_parquet("../../../data/processed/preprocessed_minimal.parquet")

In [6]:
binary_informative_df.columns

Index(['stub_id', 'particle_id', 'ac', 'ag', 'al', 'ar', 'as', 'at', 'au', 'b',
       'ba', 'bi', 'br', 'ca', 'cd', 'ce', 'cl', 'co', 'cr', 'cs', 'cu', 'dy',
       'er', 'eu', 'f', 'fe', 'fr', 'ga', 'gd', 'ge', 'hf', 'hg', 'ho', 'i',
       'in', 'ir', 'k', 'kr', 'la', 'lu', 'mg', 'mn', 'mo', 'n', 'na', 'nb',
       'nd', 'ne', 'ni', 'np', 'o', 'os', 'p', 'pa', 'pb', 'pd', 'pm', 'po',
       'pr', 'pt', 'pu', 'ra', 'rb', 're', 'rh', 'rn', 'ru', 's', 'sb', 'sc',
       'se', 'si', 'sm', 'sn', 'sr', 'ta', 'tb', 'tc', 'te', 'th', 'ti', 'tl',
       'tm', 'u', 'v', 'w', 'xe', 'y', 'yb', 'zn', 'zr', 'class', 'label',
       'target'],
      dtype='str')